# Lesson 2.4 — Infrastructure and Database Visibility

## Learning Objectives
- Understand what Infrastructure Visibility provides (Machine Agent metrics)
- Understand what Database Visibility provides (agentless DB profiling)
- Correlate infrastructure metrics to BT performance
- Navigate the Servers and Database views in the Controller UI

**Reference**: [Infrastructure Visibility](https://help.splunk.com/en/appdynamics-saas/infrastructure-visibility/26.2.0/infrastructure-visibility) | [Database Visibility](https://help.splunk.com/en/appdynamics-saas/database-visibility/26.2.0/database-visibility)

## 1. Infrastructure Visibility (Machine Agent)

The **Machine Agent** collects host-level metrics from the servers where your application runs:

| Metric Category | Metrics Collected |
|-----------------|---------|
| **CPU** | % Busy, % Idle, % User, % System, % Wait |
| **Memory** | Used (MB), Free (MB), % Used, Swap Used |
| **Disk** | I/O reads/writes per sec, % Utilization, Latency |
| **Network** | Bytes In/Out per sec, Packets per sec, Errors |
| **Processes** | Top CPU/memory consuming processes |
| **JVM** (on Java hosts) | Heap usage, GC time, Thread count, Class count |

### Why this matters:
A BT might be slow not because of bad code but because the **host is saturated**:
- CPU at 100% → all transactions slow down
- Memory pressure → JVM GC pauses → latency spikes
- Disk I/O saturated → logging writes blocking execution
- Network packet loss → external API calls timing out

In [ ]:
import random, matplotlib.pyplot as plt

random.seed(42)
minutes = list(range(60))

# Simulate an incident: memory pressure leads to GC pauses → BT latency spike
cpu_pct    = [random.gauss(40, 5) if m < 35 else random.gauss(88, 4) for m in minutes]
heap_mb    = [random.gauss(512, 30) + m * 4 for m in minutes]  # memory leak
gc_time_ms = [random.gauss(10, 3) if heap_mb[m] < 700 else random.gauss(180, 30) for m in minutes]
bt_avg_rt  = [random.gauss(300, 30) + gc_time_ms[m] * 3 for m in minutes]

fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)

axes[0].plot(minutes, cpu_pct, color='darkorange')
axes[0].axhline(80, linestyle='--', color='red', label='High CPU threshold')
axes[0].set_ylabel('CPU %')
axes[0].set_title('Infrastructure + BT Correlation: Memory Leak Scenario (60 min)')
axes[0].legend(fontsize=8)

axes[1].plot(minutes, heap_mb, color='royalblue')
axes[1].axhline(700, linestyle='--', color='red', label='High Heap threshold (700MB)')
axes[1].set_ylabel('JVM Heap (MB)')
axes[1].legend(fontsize=8)

axes[2].plot(minutes, gc_time_ms, color='purple')
axes[2].axhline(100, linestyle='--', color='red', label='High GC threshold')
axes[2].set_ylabel('GC Time (ms/min)')
axes[2].legend(fontsize=8)

axes[3].plot(minutes, bt_avg_rt, color='tomato')
axes[3].axhline(300 * 3, linestyle='--', color='orange', label='Slow threshold')
axes[3].set_ylabel('BT Avg RT (ms)')
axes[3].set_xlabel('Minute')
axes[3].legend(fontsize=8)

# Highlight GC pressure zone
for ax in axes:
    ax.axvspan(35, 60, alpha=0.1, color='red')

plt.tight_layout()
plt.savefig('infra_correlation.png', dpi=100, bbox_inches='tight')
plt.show()
print("Root cause chain: Memory leak → JVM Heap full → GC pauses → BT latency spike")

## 2. Machine Agent Extensions

The Machine Agent is extensible. Custom extensions let you collect metrics from any system:

```
Machine Agent
├── Built-in: CPU, Memory, Disk, Network
├── JVM Extension (for Java hosts)
├── Custom Extensions (from AppDynamics Exchange):
│   ├── Redis extension — queue depth, hit rate, etc.
│   ├── Nginx extension — connections, request rate
│   ├── Kafka extension — consumer lag, partition metrics
│   └── Custom Python/shell extension — any metric you define
```

**Reference**: [Extensions and Custom Metrics](https://help.splunk.com/en/appdynamics-saas/infrastructure-visibility/26.2.0/machine-agent/extensions-and-custom-metrics)

Extensions report custom metrics using the same metric path format:
```
Custom Metrics|Kafka|consumer-group-orders|lag
```

## 3. Database Visibility

**Database Visibility** gives you deep insight into database performance **without installing an agent on the database server**.

### How it works:
1. A **Database Agent** connects to your database using JDBC credentials
2. It profiles queries by sampling the database activity (active queries, wait states)
3. Data is sent to the Controller and correlated with BT call data from App Agents

### What you get:
| Feature | Description |
|---------|-------------|
| **Query Monitor** | Top queries by total time, executions, avg time |
| **Wait State Analysis** | What the DB is waiting on (lock waits, I/O, CPU) |
| **Execution Plans** | Full EXPLAIN PLAN for expensive queries |
| **BT Correlation** | Which BT triggered a specific slow query |
| **Schema Changes** | Detection of DDL changes (new indexes, table modifications) |

### Supported Databases:
Oracle, MySQL, Microsoft SQL Server, PostgreSQL, Sybase, DB2, MongoDB, Cassandra, and more.

In [ ]:
# Simulate a Database Visibility query report

import random
random.seed(5)

queries = [
    {
        "query": "SELECT o.*, oi.* FROM orders o JOIN order_items oi ON o.id = oi.order_id WHERE o.user_id = ?",
        "executions_per_min": 45.2,
        "avg_time_ms": 87.3,
        "total_time_ms_per_min": 3945.96,
        "bt": "/checkout",
        "wait_state": "CPU"
    },
    {
        "query": "SELECT p.* FROM products p WHERE p.category_id = ? ORDER BY p.created_at DESC LIMIT 20",
        "executions_per_min": 210.8,
        "avg_time_ms": 12.1,
        "total_time_ms_per_min": 2550.68,
        "bt": "/search",
        "wait_state": "CPU"
    },
    {
        "query": "UPDATE inventory SET quantity = quantity - ? WHERE product_id = ? AND quantity >= ?",
        "executions_per_min": 42.1,
        "avg_time_ms": 428.5,   # ← slow! possible lock wait
        "total_time_ms_per_min": 18040.85,
        "bt": "/checkout",
        "wait_state": "Lock Wait"  # ← the problem!
    },
    {
        "query": "INSERT INTO audit_log (event, user_id, timestamp, payload) VALUES (?, ?, ?, ?)",
        "executions_per_min": 89.3,
        "avg_time_ms": 5.2,
        "total_time_ms_per_min": 464.36,
        "bt": "multiple",
        "wait_state": "I/O"
    },
]

# Sort by total time (most expensive first)
queries.sort(key=lambda q: q['total_time_ms_per_min'], reverse=True)

print("Database Visibility — Top Queries by Total Time (per minute)")
print("=" * 100)
print(f"{'Rank':<5} {'Avg (ms)':>8} {'Exec/min':>9} {'Total (ms/min)':>15} {'Wait':>12} {'BT':<15} Query")
print("-" * 100)

for rank, q in enumerate(queries, 1):
    query_short = q['query'][:50] + '...' if len(q['query']) > 50 else q['query']
    flag = " ◄ SLOW!" if q['avg_time_ms'] > 200 else ""
    print(f"{rank:<5} {q['avg_time_ms']:>8.1f} {q['executions_per_min']:>9.1f} {q['total_time_ms_per_min']:>15.1f} "
          f"{q['wait_state']:>12} {q['bt']:<15} {query_short}{flag}")

print()
print("Finding: UPDATE inventory is suffering Lock Waits — likely a concurrency issue.")
print("Recommendation: Add SELECT FOR UPDATE SKIP LOCKED or implement optimistic locking.")

In [ ]:
# EXERCISE 2.4: Infrastructure correlation analysis
#
# You have 60 minutes of CPU and BT response time data.
# TASK: Calculate the Pearson correlation coefficient between CPU% and BT avg response time.
# A high correlation (> 0.7) suggests CPU saturation is causing latency.

import statistics

# Use the data from the chart above
# cpu_pct and bt_avg_rt are already defined

def pearson_correlation(x: list, y: list) -> float:
    """
    TASK: Compute the Pearson correlation coefficient between two lists.
    Formula: r = Σ[(xi - x̄)(yi - ȳ)] / sqrt(Σ(xi - x̄)² × Σ(yi - ȳ)²)
    """
    # TODO: Implement without using numpy
    pass

r = pearson_correlation(cpu_pct, bt_avg_rt)
if r is not None:
    print(f"Pearson correlation between CPU% and BT Response Time: r = {r:.4f}")
    if r > 0.7:
        print("Strong positive correlation — CPU saturation is likely driving BT latency.")
    elif r > 0.4:
        print("Moderate correlation — CPU may be a contributing factor.")
    else:
        print("Weak correlation — CPU is likely not the primary cause.")
else:
    print("Implement pearson_correlation() above.")

## Summary

- **Infrastructure Visibility** (Machine Agent) collects host-level metrics: CPU, memory, disk, network, JVM
- **Database Visibility** (Database Agent) profiles database queries agentlessly: top queries, wait states, execution plans
- Both layers correlate to BT performance — AppDynamics shows side-by-side comparisons in the UI
- Machine Agent extensions let you add custom metrics from any system (Redis, Kafka, Nginx, etc.)

---

**Next**: [Lesson 2.5 — End User Monitoring](lesson-2.5-end-user-monitoring.ipynb)